# Qwen-Image 2.1 image generation with OpenVINO

Qwen-Image 2.1 is a unified image generation model that supports text-to-image generation and image-conditioned editing in one pipeline. The prompt and condition images are encoded together by Qwen3-VL and processed by a block-causal diffusion transformer.

For model architecture and usage details, see the [`Qwen/Qwen-Image-2.1`](https://huggingface.co/Qwen/Qwen-Image-2.1) model card and the [Qwen-Image source repository](https://github.com/QwenLM/Qwen-Image).

This tutorial demonstrates how to:

- load a pre-exported OpenVINO IR of [Qwen-Image 2.1](https://huggingface.co/Qwen/Qwen-Image-2.1) from a local directory;
- run text-to-image generation with OpenVINO GenAI;
- run image-conditioned editing with the same exported model;
- measure pipeline loading, first-run, and warm-run latency;
- launch an interactive demo.

> **Important:** Qwen-Image 2.1 image conditioning is not classic image-to-image generation based on adding noise to an initial image. The condition image is part of the multimodal context, so this notebook intentionally does not expose a `strength` parameter.

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

> **Resource note:** The notebook loads the model from disk and keeps every component of the selected pipeline in memory. The default FP16 IR occupies roughly 44 GB on disk, so make sure the machine has enough RAM and that other memory-hungry applications are closed.

#### Table of contents:

- [Prerequisites](#Prerequisites)
    - [Installation](#Installation)
- [Select the OpenVINO model](#Select-the-OpenVINO-model)
- [Run OpenVINO GenAI inference](#Run-OpenVINO-GenAI-inference)
    - [Text-to-image](#Text-to-image)
        - [Release the text-to-image pipeline](#Release-the-text-to-image-pipeline)
    - [Image-conditioned editing](#Image-conditioned-editing)
        - [Release the image editing pipeline](#Release-the-image-editing-pipeline)
- [Benchmark](#Benchmark)
- [Interactive demo](#Interactive-demo)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen-image-2.1/qwen-image-2.1.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

This notebook loads an already exported OpenVINO IR of [`Qwen/Qwen-Image-2.1`](https://huggingface.co/Qwen/Qwen-Image-2.1) from a local directory. Nothing is downloaded from the Hugging Face Hub and no conversion runs here.

The IR has to be produced beforehand by an Optimum Intel / OpenVINO GenAI build that supports Qwen-Image 2.1, and it has to keep the component layout that `QwenImage21Pipeline` expects:

```text
<model directory>/
├── model_index.json          # must declare "_class_name": "QwenImage21Pipeline"
├── processor/                # OpenVINO tokenizer and detokenizer
├── scheduler/                # FlowMatchEulerDiscreteScheduler configuration
├── text_encoder/             # Qwen3-VL text encoder used by text-to-image
├── text_encoder_i2i/         # text encoder used by image-conditioned editing
├── transformer/              # block-causal diffusion transformer
├── vae_encoder/
├── vae_decoder/
└── vision_encoder/
```

The installation uses OpenVINO, OpenVINO Tokenizers, and OpenVINO GenAI nightly wheels. The OpenVINO GenAI build must provide the `QwenImage21Pipeline` implementation, otherwise loading fails with an unsupported pipeline error.


In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

# This notebook is offline: nothing is downloaded or installed while it runs. The shared
# helpers are imported from the repository's utils/ directory, or from a copy placed next
# to the notebook.
_here = Path.cwd()
for _candidate in (_here, *_here.parents):
    _roots = (_candidate, _candidate / "utils")
    _found = next((_root for _root in _roots if (_root / "notebook_utils.py").is_file()), None)
    if _found is not None:
        if str(_found) not in sys.path:
            sys.path.insert(0, str(_found))
        break

# Telemetry is opt-in so that offline runs never touch the network.
# Read more at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
if os.getenv("QWEN_IMAGE_21_TELEMETRY") == "1":
    from notebook_utils import collect_telemetry

    collect_telemetry("qwen-image-2.1.ipynb")

### Installation
[back to top ⬆️](#Table-of-contents:)

This notebook runs offline and installs nothing at runtime. The environment has to provide the dependencies beforehand:

- an OpenVINO GenAI build that implements `QwenImage21Pipeline` (the OpenVINO nightly wheel index is required while Qwen-Image 2.1 is experimental);
- OpenVINO and OpenVINO Tokenizers matching that GenAI build;
- Gradio, NumPy, Pillow, and ipywidgets for the interactive demo.

The exported OpenVINO IR has to exist on disk before the notebook runs; see the model directory layout in [Prerequisites](#Prerequisites).


## Select the OpenVINO model
[back to top ⬆️](#Table-of-contents:)

Set `Model root` to the directory that holds the pre-exported OpenVINO IR of [`Qwen/Qwen-Image-2.1`](https://huggingface.co/Qwen/Qwen-Image-2.1), then choose the weight precision. The OpenVINO GenAI pipelines read every component directly from disk; no model file is downloaded or converted by this notebook.

The notebook expects one directory per precision, named `Qwen-Image-2.1-IR-<precision>`:

- `FP16` → `<Model root>/Qwen-Image-2.1-IR-FP16`
- `INT4` → `<Model root>/Qwen-Image-2.1-IR-INT4`

The default root is `C:\openvino`. Use the `QWEN_IMAGE_21_OV_DIR` environment variable or edit the widget to select another root.


In [ ]:
model_id = "Qwen/Qwen-Image-2.1"
DEFAULT_MODEL_ROOT = r"C:\openvino"
PRECISIONS = ("FP16", "INT4")

export_root = widgets.Text(
    value=os.getenv("QWEN_IMAGE_21_OV_DIR", DEFAULT_MODEL_ROOT),
    description="Model root:",
    placeholder="Directory containing the per-precision OpenVINO IR directories",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "120px"},
)
weight_format = widgets.Dropdown(
    options=PRECISIONS,
    value="FP16",
    description="Weights:",
    style={"description_width": "120px"},
)

print(f"Source model: https://huggingface.co/{model_id}")
display(export_root, weight_format)


In [ ]:
model_dir = Path(export_root.value).expanduser() / f"Qwen-Image-2.1-IR-{weight_format.value}"
precision_label = weight_format.value

required_components = (
    "transformer",
    "text_encoder",
    "text_encoder_i2i",
    "vae_encoder",
    "vae_decoder",
    "vision_encoder",
    "processor",
    "scheduler",
)

if not model_dir.is_dir():
    raise FileNotFoundError(f"The OpenVINO model directory {model_dir} does not exist. Expected <Model root>/Qwen-Image-2.1-IR-{weight_format.value}. Set QWEN_IMAGE_21_OV_DIR or edit the Model root widget.")

if not (model_dir / "model_index.json").is_file():
    raise FileNotFoundError(f"{model_dir} does not contain model_index.json, so it is not an OpenVINO GenAI image model directory.")

missing_components = [component for component in required_components if not (model_dir / component).is_dir()]
if missing_components:
    raise RuntimeError(f"{model_dir} is missing components required by QwenImage21Pipeline: {', '.join(missing_components)}")

print(f"Hugging Face source model: {model_id}")
print(f"OpenVINO model directory: {model_dir}")
print(f"Precision: {precision_label}")


## Run OpenVINO GenAI inference
[back to top ⬆️](#Table-of-contents:)

The local OpenVINO IR directory is used by both OpenVINO GenAI pipelines. Qwen-Image 2.1 defaults are used throughout the notebook:

- 40 denoising steps;
- guidance scale 1.0;
- no negative prompt;
- 1024 x 1024 output;
- fixed random seed.

The pipeline-specific construction and generation calls are kept in isolated cells so the T2I and image-editing lifecycles remain explicit.


In [6]:
import gc
import time
from datetime import datetime
from typing import Any

import numpy as np
import openvino as ov
import openvino_genai as ov_genai
from PIL import Image, ImageOps
from tqdm.auto import tqdm

from notebook_utils import device_widget


def image_to_tensor(image: Image.Image) -> ov.Tensor:
    """Convert a PIL image to an NHWC OpenVINO tensor."""
    image_data = np.asarray(image.convert("RGB"), dtype=np.uint8)[None]
    return ov.Tensor(image_data)


def output_to_image(output: Any) -> Image.Image:
    """Convert an OpenVINO GenAI image output to a PIL image."""
    data = output.data if hasattr(output, "data") else output
    array = np.asarray(data)
    if array.ndim == 4:
        array = array[0]
    return Image.fromarray(array).convert("RGB")


def make_image_comparison(source_image: Image.Image, generated_image: Image.Image) -> Image.Image:
    """Place source and generated images side by side."""
    source_preview = ImageOps.fit(source_image, generated_image.size, method=Image.Resampling.LANCZOS)
    comparison = Image.new("RGB", (generated_image.width * 2, generated_image.height))
    comparison.paste(source_preview, (0, 0))
    comparison.paste(generated_image, (generated_image.width, 0))
    return comparison


def save_generated_image(image: Image.Image, scenario: str, precision: str, inference_device: str, seed: int) -> Path:
    """Save a generated image with its inference configuration in the filename."""
    output_dir = Path("generated_images")
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    safe_device = inference_device.replace(":", "-").replace(".", "-")
    output_path = output_dir / f"qwen-image-2.1-{scenario}-{precision.lower()}-{safe_device.lower()}-seed{seed}-{timestamp}.png"
    image.save(output_path)
    print(f"Saved image to {output_path.resolve()}")
    return output_path


def generate_with_progress(
    pipeline: Any,
    *args: Any,
    num_inference_steps: int,
    description: str,
    **kwargs: Any,
) -> Any:
    """Run image generation while displaying denoising progress."""
    progress = tqdm(total=num_inference_steps, desc=description)

    def callback(step: int, num_steps: int, latent: Any) -> bool:
        completed_steps = min(step + 1, num_steps)
        progress.update(max(0, completed_steps - progress.n))
        return False

    try:
        return pipeline.generate(
            *args,
            num_inference_steps=num_inference_steps,
            callback=callback,
            **kwargs,
        )
    finally:
        progress.close()


def print_perf_metrics(output: Any) -> None:
    """Print available OpenVINO GenAI image performance metrics."""
    metrics = getattr(output, "perf_metrics", None)
    if metrics is None:
        print("This OpenVINO GenAI image output does not expose perf_metrics; wall-clock metrics are used.")
        return

    metric_methods = {
        "Load time": "get_load_time",
        "Generate duration": "get_generate_duration",
        "Transformer duration": "get_transformer_infer_duration",
        "VAE encoder duration": "get_vae_encoder_infer_duration",
        "VAE decoder duration": "get_vae_decoder_infer_duration",
    }
    for label, method_name in metric_methods.items():
        method = getattr(metrics, method_name, None)
        if method is not None:
            print(f"{label}: {method()}")


device = device_widget(default="CPU", exclude=["NPU"])
device

Dropdown(description='Device:', options=('CPU', 'GPU.0', 'GPU.1', 'AUTO'), value='CPU')

### Text-to-image
[back to top ⬆️](#Table-of-contents:)

Load the text-to-image pipeline and record model loading time separately from generation latency.


In [7]:
t2i_load_started = time.perf_counter()
t2i_pipe = ov_genai.Text2ImagePipeline(str(model_dir), device.value)
t2i_load_seconds = time.perf_counter() - t2i_load_started

print(f"Text-to-image pipeline load time: {t2i_load_seconds:.2f} s")

Text-to-image pipeline load time: 12.59 s


In [ ]:
t2i_prompt = 'A cozy coffee shop with a chalkboard sign reading "Qwen Coffee", cinematic lighting'
t2i_num_inference_steps = 40
t2i_seed = 21

t2i_first_started = time.perf_counter()
t2i_output = generate_with_progress(
    t2i_pipe,
    t2i_prompt,
    height=1024,
    width=1024,
    guidance_scale=1.0,
    num_inference_steps=t2i_num_inference_steps,
    description="Text-to-image",
    generator=ov_genai.TorchGenerator(t2i_seed),
)
t2i_first_run_seconds = time.perf_counter() - t2i_first_started

print(f"First text-to-image generation: {t2i_first_run_seconds:.2f} s")
print_perf_metrics(t2i_output)
t2i_image = output_to_image(t2i_output)
t2i_output_path = save_generated_image(t2i_image, "t2i", precision_label, device.value, t2i_seed)
t2i_image

#### Release the text-to-image pipeline
[back to top ⬆️](#Table-of-contents:)

Release the compiled text-to-image pipeline before loading the image editing pipeline.


In [9]:
if "t2i_pipe" in globals():
    del t2i_pipe
    gc.collect()
    print("Text-to-image pipeline released.")
else:
    print("Text-to-image pipeline is not loaded.")

Text-to-image pipeline released.


### Image-conditioned editing
[back to top ⬆️](#Table-of-contents:)

Select a local condition image and describe the requested edit. The image is multimodal context for Qwen-Image 2.1; it is not a noisy initialization controlled by `strength`.

`Image2ImagePipeline` receives the condition tensor as the second positional argument. Height and width are intentionally omitted: the Qwen-Image 2.1 pipeline derives a 32-aligned output resolution near 1024² pixels from the condition image aspect ratio.


In [10]:
sample_image_path = Path(os.getenv("QWEN_IMAGE_21_INPUT_IMAGE", "sample_cat.png")).expanduser()
if not sample_image_path.is_file():
    raise FileNotFoundError(
        f"Condition image {sample_image_path} was not found. Place a local image next to the notebook "
        "or set QWEN_IMAGE_21_INPUT_IMAGE to an existing image path."
    )

input_image_path = widgets.Text(
    value=str(sample_image_path),
    description="Input image:",
    placeholder="Path to a local image used for editing",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "120px"},
)
input_image_path

Text(value='sample_cat.png', description='Input image:', layout=Layout(width='90%'), placeholder='Path to a lo…

In [ ]:
condition_image_path = Path(input_image_path.value).expanduser()
if not condition_image_path.is_file():
    raise FileNotFoundError("Set QWEN_IMAGE_21_INPUT_IMAGE or the Input image widget to a local image file.")

condition_image = Image.open(condition_image_path).convert("RGB")
condition_tensor = image_to_tensor(condition_image)
edit_prompt = "Put a tiny red top hat and a blue bow tie on the cat. Preserve the cat's identity, face, pose, and the original composition."
i2i_num_inference_steps = 40
i2i_seed = 42

i2i_load_started = time.perf_counter()
i2i_pipe = ov_genai.Image2ImagePipeline(str(model_dir), device.value)
i2i_load_seconds = time.perf_counter() - i2i_load_started

i2i_first_started = time.perf_counter()
i2i_output = generate_with_progress(
    i2i_pipe,
    edit_prompt,
    condition_tensor,
    guidance_scale=1.0,
    num_inference_steps=i2i_num_inference_steps,
    description="Image editing",
    generator=ov_genai.TorchGenerator(i2i_seed),
)
i2i_first_run_seconds = time.perf_counter() - i2i_first_started

print(f"Image editing pipeline load time: {i2i_load_seconds:.2f} s")
print(f"First image editing generation: {i2i_first_run_seconds:.2f} s")
print_perf_metrics(i2i_output)
i2i_image = output_to_image(i2i_output)
i2i_output_path = save_generated_image(i2i_image, "editing", precision_label, device.value, i2i_seed)
i2i_comparison = make_image_comparison(condition_image, i2i_image)
i2i_comparison

#### Release the image editing pipeline
[back to top ⬆️](#Table-of-contents:)

Release the compiled image editing pipeline before running benchmarks or launching the interactive demo.


In [12]:
if "i2i_pipe" in globals():
    del i2i_pipe
    gc.collect()
    print("Image editing pipeline released.")
else:
    print("Image editing pipeline is not loaded.")

Image editing pipeline released.


## Benchmark
[back to top ⬆️](#Table-of-contents:)

The benchmark reports end-to-end pipeline latency rather than isolated OpenVINO IR component performance.

- Pipeline loading and the first generation are measured in this benchmark cell.
- The first generation serves as the excluded warm-up run.
- Warm latency reports minimum, median, mean, standard deviation, and individual measurements.
- T2I and image-conditioned editing are loaded, measured, and released sequentially.
- The report includes the actual output resolution, device, precision, inference settings, and cold total (`load + first generation`).

For reproducible comparisons, keep the device, precision, resolution, number of steps, guidance scale, prompt, condition image, and KV-cache behavior unchanged. Close other resource-intensive applications before collecting final numbers.

Run the next cell, select the number of measured warm runs, and then run the benchmark cell below it. Five runs are used by default to make occasional slow runs visible without making this expensive benchmark excessively long.


In [13]:
benchmark_runs = widgets.IntSlider(
    value=5,
    min=3,
    max=10,
    step=1,
    description="Warm runs:",
    style={"description_width": "100px"},
)
benchmark_runs

IntSlider(value=5, description='Warm runs:', max=10, min=3, style=SliderStyle(description_width='100px'))

In [ ]:
from statistics import mean, median, pstdev
from typing import Callable


def benchmark_generation(generate: Callable[[], Any], runs: int) -> tuple[dict[str, Any], Any]:
    """Measure repeated generation latency and return the last output."""
    durations = []
    output = None
    for _ in range(runs):
        started = time.perf_counter()
        output = generate()
        durations.append(time.perf_counter() - started)

    return {
        "minimum": min(durations),
        "median": median(durations),
        "mean": mean(durations),
        "stddev": pstdev(durations),
        "durations": durations,
    }, output


BENCHMARK_STEPS = 40
BENCHMARK_GUIDANCE_SCALE = 1.0


def generate_t2i_benchmark(pipeline: Any) -> Any:
    """Generate the text-to-image benchmark sample."""
    return pipeline.generate(
        t2i_prompt,
        height=1024,
        width=1024,
        guidance_scale=BENCHMARK_GUIDANCE_SCALE,
        num_inference_steps=BENCHMARK_STEPS,
        generator=ov_genai.TorchGenerator(t2i_seed),
    )


def generate_i2i_benchmark(pipeline: Any) -> Any:
    """Generate the image-editing benchmark sample."""
    return pipeline.generate(
        edit_prompt,
        condition_tensor,
        guidance_scale=BENCHMARK_GUIDANCE_SCALE,
        num_inference_steps=BENCHMARK_STEPS,
        generator=ov_genai.TorchGenerator(i2i_seed),
    )


t2i_load_started = time.perf_counter()
t2i_benchmark_pipe = ov_genai.Text2ImagePipeline(str(model_dir), device.value)
t2i_benchmark_load_seconds = time.perf_counter() - t2i_load_started
try:
    t2i_first_started = time.perf_counter()
    t2i_first_output = generate_t2i_benchmark(t2i_benchmark_pipe)
    t2i_benchmark_first_seconds = time.perf_counter() - t2i_first_started
    del t2i_first_output

    t2i_warm, t2i_benchmark_output = benchmark_generation(
        lambda: generate_t2i_benchmark(t2i_benchmark_pipe),
        benchmark_runs.value,
    )
    t2i_benchmark_size = output_to_image(t2i_benchmark_output).size
    del t2i_benchmark_output
finally:
    del t2i_benchmark_pipe
    gc.collect()

i2i_load_started = time.perf_counter()
i2i_benchmark_pipe = ov_genai.Image2ImagePipeline(str(model_dir), device.value)
i2i_benchmark_load_seconds = time.perf_counter() - i2i_load_started
try:
    i2i_first_started = time.perf_counter()
    i2i_first_output = generate_i2i_benchmark(i2i_benchmark_pipe)
    i2i_benchmark_first_seconds = time.perf_counter() - i2i_first_started
    del i2i_first_output

    i2i_warm, i2i_benchmark_output = benchmark_generation(
        lambda: generate_i2i_benchmark(i2i_benchmark_pipe),
        benchmark_runs.value,
    )
    i2i_benchmark_size = output_to_image(i2i_benchmark_output).size
    del i2i_benchmark_output
finally:
    del i2i_benchmark_pipe
    gc.collect()

benchmark_results = {
    "Text-to-image": {
        "resolution": t2i_benchmark_size,
        "load": t2i_benchmark_load_seconds,
        "first": t2i_benchmark_first_seconds,
        **t2i_warm,
    },
    "Image editing": {
        "resolution": i2i_benchmark_size,
        "load": i2i_benchmark_load_seconds,
        "first": i2i_benchmark_first_seconds,
        **i2i_warm,
    },
}

print("Benchmark configuration:")
print(f"  Device: {device.value}")
print(f"  Precision: {precision_label}")
print(f"  Inference steps: {BENCHMARK_STEPS}")
print(f"  Guidance scale: {BENCHMARK_GUIDANCE_SCALE}")
print("  Warm-up runs: 1 (first generation)")
print(f"  Measured warm runs: {benchmark_runs.value}")
print(f"  Editing condition: {condition_image.width}x{condition_image.height}\n")

print(
    f"{'Scenario':<18} {'Resolution':>12} {'Load (s)':>9} {'First (s)':>10} "
    f"{'Cold total (s)':>14} {'Min (s)':>9} {'Median (s)':>11} "
    f"{'Mean (s)':>10} {'Std (s)':>9}"
)
for scenario, values in benchmark_results.items():
    resolution = f"{values['resolution'][0]}x{values['resolution'][1]}"
    cold_total = values["load"] + values["first"]
    print(
        f"{scenario:<18} {resolution:>12} {values['load']:>9.2f} "
        f"{values['first']:>10.2f} {cold_total:>14.2f} "
        f"{values['minimum']:>9.2f} {values['median']:>11.2f} "
        f"{values['mean']:>10.2f} {values['stddev']:>9.2f}"
    )

print("\nIndividual warm runs:")
for scenario, values in benchmark_results.items():
    samples = ", ".join(f"{duration:.2f}" for duration in values["durations"])
    print(f"  {scenario}: {samples} s")

editing_to_t2i_ratio = benchmark_results["Image editing"]["median"] / benchmark_results["Text-to-image"]["median"]
print(f"\nImage editing / T2I warm median ratio: {editing_to_t2i_ratio:.2f}x")

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

The demo allows selecting Text-to-Image or Image Editing, FP16 or INT4 precision, and an available OpenVINO device. It keeps only the selected pipeline in memory and releases it when the configuration changes.


In [ ]:
from gradio_helper import make_demo

for pipeline_name in ("t2i_pipe", "i2i_pipe"):
    if pipeline_name in globals():
        del globals()[pipeline_name]
gc.collect()

demo = make_demo(
    model_root=export_root.value,
    default_precision=weight_format.value,
    default_device=device.value,
    editing_sample=sample_image_path,
)

# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/
demo.launch(debug=True)
